## Show, Attend and Tell — Training (Flickr8k)


Clone repo first

In [2]:
!git clone https://github.com/seanzhangw/show-attend-tell.git

Cloning into 'show-attend-tell'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 31 (delta 3), reused 26 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 13.99 KiB | 13.99 MiB/s, done.
Resolving deltas: 100% (3/3), done.


Pull recent changes

In [3]:
%cd /content/show-attend-tell
!git pull

/content/show-attend-tell
Already up to date.


Get dataset

In [4]:
%cd /content/show-attend-tell/data
!python get_flickr8k.py

/content/show-attend-tell/data
Traceback (most recent call last):
  File "/usr/lib/python3.12/http/client.py", line 1450, in getresponse
    response.begin()
  File "/usr/lib/python3.12/http/client.py", line 336, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 297, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/show-attend-tell/data/get_flickr8k.py", line 10, in <module>
    path = kagglehub.dataset_download("adityajn105/flickr8k")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-pack

In [5]:
%cd /content/show-attend-tell/code

# =========================
# 1) Imports
# =========================
import os
import time

import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

import config
from datasets.flickr8k import collate_fn, build_flickr8k_dataset_split
from eval.bleu import compute_bleu4
from eval.greedy_decode import greedy_decode
from models.encoder import EncoderCNN
from models.decoder import Decoder


/content/show-attend-tell/code


In [6]:
# =========================
# 2) Loss helpers
# =========================

def compute_caption_loss(logits, captions, criterion):
    """
    Args:
        logits:   (B, T-1, V)
        captions: (B, T)

    Returns:
        ce_loss: scalar
    """
    targets = captions[:, 1:]  # (B, T-1)
    bsz, time_steps, vocab_size = logits.shape

    logits_flat = logits.reshape(bsz * time_steps, vocab_size)   # (B*(T-1), V)
    targets_flat = targets.reshape(bsz * time_steps)             # (B*(T-1))

    return criterion(logits_flat, targets_flat)


In [7]:
# =========================
# 3) Train / Validate
# =========================

def train_one_epoch(
    encoder,
    decoder,
    loader,
    criterion,
    optimizer,
    device,
    lambda_att=1.0,
    grad_clip=5.0,
    print_every=100,
    freeze_encoder=True,
):
    """One training epoch. Returns average total loss."""

    if freeze_encoder:
        encoder.eval()
    else:
        encoder.train()
    decoder.train()

    running_loss = 0.0
    t0 = time.time()

    for batch_idx, (images, captions) in enumerate(loader, start=1):
        images = images.to(device)    # (B, 3, 224, 224)
        captions = captions.to(device)  # (B, T)

        optimizer.zero_grad()

        if freeze_encoder:
            with torch.no_grad():
                features = encoder(images)  # (B, L, D)
        else:
            features = encoder(images)      # (B, L, D)

        logits, alphas = decoder(features, captions)  # (B, T-1, V), (B, T-1, L)

        ce_loss = compute_caption_loss(logits, captions, criterion)

        # Doubly stochastic attention penalty:
        # For each location i, sum_t alpha_ti should be close to 1.
        attention_penalty = ((1.0 - alphas.sum(dim=1)) ** 2).mean()

        loss = ce_loss + lambda_att * attention_penalty
        loss.backward()

        if grad_clip is not None:
            clip_grad_norm_(decoder.parameters(), grad_clip)
            if not freeze_encoder:
                clip_grad_norm_(encoder.parameters(), grad_clip)

        optimizer.step()

        running_loss += loss.item()

        # ---- Throughput / ETA (tiny print) ----
        if batch_idx % print_every == 0:
            elapsed = time.time() - t0
            it_per_sec = batch_idx / max(elapsed, 1e-8)
            sec_per_it = 1.0 / max(it_per_sec, 1e-8)
            eta_sec = (len(loader) - batch_idx) * sec_per_it

            avg_so_far = running_loss / batch_idx
            print(
                f"[Train] {batch_idx:>5}/{len(loader)} "
                f"loss={avg_so_far:.4f} (CE={ce_loss.item():.4f}, Att={attention_penalty.item():.4f}) | "
                f"{it_per_sec:.2f} it/s | ETA {eta_sec/60:.1f} min"
            )

    return running_loss / max(1, len(loader))


@torch.no_grad()
def validate(encoder, decoder, loader, criterion, device, lambda_att=1.0):
    """Validation epoch. Returns average total loss."""

    encoder.eval()
    decoder.eval()

    running_loss = 0.0

    for images, captions in loader:
        images = images.to(device)
        captions = captions.to(device)

        features = encoder(images)              # (B, L, D)
        logits, alphas = decoder(features, captions)  # (B, T-1, V), (B, T-1, L)

        ce_loss = compute_caption_loss(logits, captions, criterion)
        attention_penalty = ((1.0 - alphas.sum(dim=1)) ** 2).mean()
        loss = ce_loss + lambda_att * attention_penalty

        running_loss += loss.item()

    return running_loss / max(1, len(loader))


In [9]:
# =========================
# 4) Setup (data, model, optimizer)
# =========================

# Hyperparameters (edit as needed)
epochs = 8
batch_size = config.BATCH_SIZE
lr = 1e-4
embed_dim = 512
hidden_dim = 512
attention_dim = 512

dropout = 0.5
lambda_att = 1.0
grad_clip = 5.0
freeze_encoder = True
val_ratio = 0.1
split_seed = 42
# BLEU-4: cap images per eval for speed (None = all val images)
bleu_max_images = 500

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ImageNet normalization for ResNet
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

# Dataset: image-level train/val split; vocab from train captions only
(
    train_set,
    val_set,
    val_image_ids,
    full_captions_map,
    word2idx,
    idx2word,
) = build_flickr8k_dataset_split(
    config,
    transform=transform,
    val_ratio=val_ratio,
    seed=split_seed,
)
pad_idx = word2idx["<pad>"]
print(
    f"Train samples: {len(train_set)} | Val samples: {len(val_set)} | "
    f"Val images: {len(val_image_ids)} | Vocab size: {len(word2idx)}"
)

train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_set,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
)

# Models
encoder = EncoderCNN().to(device)
decoder = Decoder(
    vocab_size=len(word2idx),
    embed_dim=embed_dim,
    feature_dim=2048,
    hidden_dim=hidden_dim,
    attention_dim=attention_dim,
    dropout=dropout,
).to(device)

# Loss / Optimizer
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

params = list(decoder.parameters())
if not freeze_encoder:
    params += [p for p in encoder.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=lr)


Using device: cuda
Train samples: 36410 | Val samples: 4045 | Val images: 809 | Vocab size: 8363


In [8]:
# =========================
# 5) Train (skipped — use next cell: 5b for checkpoints + BLEU-4)
# =========================
pass



Epoch 1/10
[Train]   100/1138 loss=7.1107 (CE=5.1729, Att=0.3777) | 3.02 it/s | ETA 5.7 min
[Train]   200/1138 loss=6.3257 (CE=5.3217, Att=0.3767) | 3.07 it/s | ETA 5.1 min
[Train]   300/1138 loss=5.9753 (CE=5.0055, Att=0.3769) | 3.08 it/s | ETA 4.5 min
[Train]   400/1138 loss=5.7529 (CE=4.7236, Att=0.3763) | 3.07 it/s | ETA 4.0 min
[Train]   500/1138 loss=5.5924 (CE=4.3179, Att=0.3762) | 3.07 it/s | ETA 3.5 min
[Train]   600/1138 loss=5.4637 (CE=4.3142, Att=0.3763) | 3.06 it/s | ETA 2.9 min
[Train]   700/1138 loss=5.3559 (CE=4.6000, Att=0.3768) | 3.05 it/s | ETA 2.4 min
[Train]   800/1138 loss=5.2682 (CE=4.3762, Att=0.3762) | 3.05 it/s | ETA 1.8 min


KeyboardInterrupt: 

**Testing**


In [10]:
# =========================
# 5b) Train + best checkpoint by val loss
# (Run this cell instead of the older train cell above)
# =========================

import os

checkpoint_dir = "./checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

best_val_loss = float("inf")
best_ckpt_path = os.path.join(checkpoint_dir, "best_by_val_loss.pt")

for epoch in range(1, epochs + 1):
    print(f"\nEpoch {epoch}/{epochs}")

    train_loss = train_one_epoch(
        encoder=encoder,
        decoder=decoder,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        lambda_att=lambda_att,
        grad_clip=grad_clip,
        print_every=100,
        freeze_encoder=freeze_encoder,
    )

    val_loss = validate(
        encoder=encoder,
        decoder=decoder,
        loader=val_loader,
        criterion=criterion,
        device=device,
        lambda_att=lambda_att,
    )

    print(f"[Epoch {epoch}] train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    bleu4 = compute_bleu4(
        encoder,
        decoder,
        val_image_ids,
        full_captions_map,
        config.IMAGE_DIR,
        transform,
        word2idx,
        idx2word,
        device,
        max_len=config.MAX_LEN,
        max_images=bleu_max_images,
    )
    print(
        f"[Epoch {epoch}] BLEU-4 (greedy, max {bleu_max_images or 'all'} val images): {bleu4:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "epoch": epoch,
                "val_loss": val_loss,
                "train_loss": train_loss,
                "encoder_state_dict": encoder.state_dict(),
                "decoder_state_dict": decoder.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "word2idx": word2idx,
                "idx2word": idx2word,
                "config": {
                    "embed_dim": embed_dim,
                    "hidden_dim": hidden_dim,
                    "attention_dim": attention_dim,
                    "dropout": dropout,
                    "lambda_att": lambda_att,
                    "freeze_encoder": freeze_encoder,
                },
            },
            best_ckpt_path,
        )
        print(f"  -> Saved new best checkpoint: {best_ckpt_path} (val_loss={best_val_loss:.4f})")

print(f"\nBest val_loss: {best_val_loss:.4f}")
print(f"Best checkpoint path: {best_ckpt_path}")



Epoch 1/8
[Train]   100/1138 loss=7.0313 (CE=5.2851, Att=0.3775) | 3.10 it/s | ETA 5.6 min
[Train]   200/1138 loss=6.2815 (CE=5.1750, Att=0.3767) | 3.12 it/s | ETA 5.0 min
[Train]   300/1138 loss=5.9440 (CE=4.8568, Att=0.3766) | 3.12 it/s | ETA 4.5 min
[Train]   400/1138 loss=5.7262 (CE=4.5644, Att=0.3765) | 3.12 it/s | ETA 3.9 min
[Train]   500/1138 loss=5.5650 (CE=4.6441, Att=0.3762) | 3.12 it/s | ETA 3.4 min
[Train]   600/1138 loss=5.4382 (CE=4.3397, Att=0.3765) | 3.12 it/s | ETA 2.9 min
[Train]   700/1138 loss=5.3354 (CE=4.3313, Att=0.3762) | 3.11 it/s | ETA 2.3 min
[Train]   800/1138 loss=5.2458 (CE=4.4744, Att=0.3764) | 3.12 it/s | ETA 1.8 min
[Train]   900/1138 loss=5.1728 (CE=4.3929, Att=0.3764) | 3.12 it/s | ETA 1.3 min
[Train]  1000/1138 loss=5.1096 (CE=4.4068, Att=0.3766) | 3.12 it/s | ETA 0.7 min
[Train]  1100/1138 loss=5.0500 (CE=3.6632, Att=0.3767) | 3.12 it/s | ETA 0.2 min
[Epoch 1] train_loss=5.0292 | val_loss=4.3498
[Epoch 1] BLEU-4 (greedy, max 500 val images): 0.117

In [10]:
# =========================
# 6) Qualitative eval (greedy decoding on ~20 val images)
# =========================
# Uses image-level val split: one decode per image vs all references in full_captions_map.

from PIL import Image

num_examples = min(20, len(val_image_ids))
print(f"\nQualitative eval on {num_examples} validation images (unique)")

for i, img_name in enumerate(sorted(val_image_ids)[:num_examples]):
    path = os.path.join(config.IMAGE_DIR, img_name)
    image = Image.open(path).convert("RGB")
    image_tensor = transform(image)

    pred_tokens = greedy_decode(
        encoder,
        decoder,
        image_tensor,
        word2idx,
        idx2word,
        device,
        max_len=config.MAX_LEN,
    )
    refs = full_captions_map[img_name][:5]

    print(f"\n[{i+1:02d}] Image: {img_name}")
    print("Generated:", " ".join(pred_tokens) if pred_tokens else "<empty>")
    for j, ref in enumerate(refs[:3], start=1):
        print(f"Ref {j}: {ref}")



Qualitative eval on 20 validation images

[01] Image: 756004341_1a816df714.jpg
Generated: two children wearing sunglasses and sunglasses
Ref 1: two children in a yard wearing sunglasses , with a play structure behind them .
Ref 2: two children in orange sunglasses .
Ref 3: two children wearing sunglasses sit outside of a plastic tent .

[02] Image: 2276120079_4f235470bc.jpg
Generated: two children are running through a grassy field with a small child in a green shirt
Ref 1: a boy runs as others play on a home-made slip and slide .
Ref 2: children in swimming clothes in a field .
Ref 3: little kids are playing outside with a water hose and are sliding down a water slide .

[03] Image: 241346580_b3c035d65c.jpg
Generated: a football player in a red uniform is running in a field
Ref 1: a group of men play a college football game .
Ref 2: a red team and a white team are playing football .
Ref 3: football players avoiding a fumble .

[04] Image: 3259002340_707ce96858.jpg
Generated: a black 

In [12]:
from google.colab import drive
import sys

# TODO 0: Mount your Google Drive; this allows the runtime environment to access your drive.
drive.mount('/content/gdrive')

# NOTE: Make sure your path does NOT include a '/' at the end!
base_dir = "/content/gdrive/MyDrive/show-attend-tell"
sys.path.append(base_dir)
## END TODO

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [13]:
import shutil
import os

# Define the source (where the model is now) and destination (your Drive)
source_path = './checkpoints/best_by_val_loss.pt'
dest_path = os.path.join(base_dir, 'best_by_val_loss.pt')

# Create the directory on Drive if it doesn't exist
os.makedirs(base_dir, exist_ok=True)

# Copy the file
shutil.copyfile(source_path, dest_path)

print(f"Model successfully saved to: {dest_path}")

Model successfully saved to: /content/gdrive/MyDrive/show-attend-tell/best_by_val_loss.pt
